# 05_data_quality — Pipeline Validation Gate

Runs after gold. **Fails the job** if any check fails.

**Extended this revision:** the original version only checked row counts, 
nulls, and Article_ID cross-table consistency. New checks below validate 
the actual *mathematical outputs* of `04_gold_aggregation`'s TR-11–TR-14 
logic — trend score ranges, that decay is genuinely being applied (not a 
silent no-op), Engagement's null-vs-numeric split by source category, and 
bidirectional orphan-record checks between silver and gold.

In [0]:
%run ./00_config

# 00_config — Shared Config & Helpers

`%run` by every other notebook. Centralizes the storage connection string, 
source list, table/volume names, and small logging/counting helpers.

**This is the connection-string variant of the pipeline.** Bronze reads 
files landed in ADLS using the plain Python `azure-storage-blob` SDK and a 
storage account connection string — not Auto Loader over an `abfss://` 
path, and not a Unity Catalog External Location. That's a deliberate 
choice, not a step backward: Databricks Serverless blocks 
`spark.conf.set("fs.azure.account.key...")` outright (`CONFIG_NOT_AVAILABLE`), 
which is what Auto Loader over `abfss://` would need without an External 
Location, and the External Location route needs grants (`READ FILES`, 
`WRITE FILES`, `CREATE MANAGED STORAGE`) that may not be available to you 
on a shared/class storage account. The Blob SDK never touches Spark's 
filesystem config at all -- it's a plain HTTPS call, same mechanism the 
live-fetch version (`py_work/`) already uses successfully against public 
APIs on serverless -- so it works regardless of either restriction.

**Note on architecture:** Bronze/Silver/Gold are **Unity Catalog managed 
tables** — `fomoless.bronze.articles_raw`, `fomoless.silver.articles`, 
`fomoless.silver.articles_rejected`, `fomoless.gold.articles`. `01_setup` 
creates the schemas (assumes the `fomoless` catalog itself already exists — 
see the note in `01_setup` if it doesn't yet).

**Bronze is a plain batch step, not a stream** — since there's no Auto 
Loader checkpoint tracking "what's already been read", incrementality 
comes from checking each landed file's blob name against what's already in 
`bronze.articles_raw` (`source_blob_name`) before downloading it, so a 
re-run only pulls genuinely new files. **Silver stays incremental** via its 
own Structured Streaming checkpoint, stored in a small Unity Catalog 
**managed Volume** (`VOLUME_PATH`, created by `01_setup`) — Databricks-managed 
storage under the catalog you already own, so it needs no extra grants 
either. That Volume also holds nothing else; dedup-merge audit logs go 
straight to ADLS via the same Blob SDK connection as bronze.

---
**SECURITY FIX (this revision):** a previous version of this notebook had 
a live Azure Storage account key hardcoded as the `connection_string` 
widget's default value. That key has been removed from this file — **if 
you're seeing this note, rotate that key in Azure Portal (Storage Account 
→ Access keys → Rotate) if you haven't already**, since it was exposed in 
a shared notebook. Going forward, set the connection string via the widget 
at run time or the `AZURE_STORAGE_CONNECTION_STRING` environment variable 
— never as a hardcoded default.

**New in this revision:** `CANONICAL_SOURCE_NAMES` (CHK-4 enum gate), 
`SOURCES_WITHOUT_NATIVE_SUMMARY` (CHK-5 conditional-completeness exception 
list), and `CLOCK_SKEW_LOG_PREFIX` (CHK-7 audit trail) — all consumed by 
`03_silver_transform`.

---
**Gold/Mongo revision:** added `ensure_cdf_enabled` (moved here from 
`03_silver_transform` since `04_gold_aggregation` now needs the identical 
logic — shared, not duplicated), `TREND_DECAY_HALF_LIFE_DAYS` and 
`MOMENTUM_MULTIPLIERS` (TR-11/TR-13), new `gold`/`mongo_sink` checkpoint 
paths, and `get_mongo_uri()` (mirrors `get_widget_or_env` — never 
hardcoded).

**⚠️ FLAGGED GAP — TR-13's momentum multiplier table is incomplete.** The 
source Transformation Rules sheet is cut off after the `BBC`/`The Guardian` 
→ `3.0` branch; the multiplier values for dev.to, Medium, Towards Data 
Science, Hacker News, Stack Overflow Blog, Stack Exchange, and KDnuggets 
were never visible in the source image. `MOMENTUM_MULTIPLIERS` below only 
has the one confirmed entry — every other source falls back to 
`DEFAULT_MOMENTUM_MULTIPLIER = 1.0` as an explicit placeholder. **This dict 
needs real values from the business before `04_gold_aggregation`'s trend 
scores are correct**, not just structurally complete.

## 1. Serverless "spark not initialized" workaround

On some Databricks Serverless sessions, the notebook-provided `spark` 
global isn't ready the first time a cell references it, raising 
`SystemError: Internal error: spark should be initialized with the first 
notebook command.` Explicitly building a session via `databricks.connect` 
fixes it and is a no-op anywhere `spark` is already fine.

## 2. Storage connection string

From Azure Portal → your storage account → **Access keys** → Connection 
string (key 1 or key 2, either works). Settable via a Databricks widget 
(`connection_string`) or the `AZURE_STORAGE_CONNECTION_STRING` environment 
variable -- never hardcode it directly in a cell.

## 3. Sources & schema

## 4. Catalog, schemas, tables & volume

## 5. Gold trend-scoring constants (TR-11, TR-13)

## 6. MongoDB connection (for `06_load_mongodb`)

Same pattern as the storage connection string above: widget first, then 
environment variable, never a hardcoded default. `MONGO_URI` should 
already contain the target database name (e.g. 
`.../fomoless?retryWrites=true&w=majority`); `MONGO_COLLECTION_NAME` names 
the collection within it.

## 7. Helpers

[2026-09-25 20:37:47 UTC] Config loaded — container: fomoless, catalog: fomoless


In [0]:
from pyspark.sql import functions as F

bronze = spark.table(TABLES["bronze"])
silver = spark.table(TABLES["silver"])
gold = spark.table(TABLES["gold"])

n_bronze, n_silver, n_gold = bronze.count(), silver.count(), gold.count()

silver_ids = silver.select("Article_ID").distinct()
gold_ids = gold.select("Article_ID").distinct()
silver_ids_missing_from_gold = silver_ids.subtract(gold_ids).count()
# New: the reverse direction -- a gold row with no matching silver
# Article_ID would mean gold picked up something CDF shouldn't have been
# able to produce (a real orphan / integrity bug), not an expected state.
gold_ids_orphaned = gold_ids.subtract(silver_ids).count()

## Original checks (row counts, nulls, cross-table consistency)

In [0]:
checks = [
    ("bronze has rows",                          n_bronze >= DQ_MIN_ROWS,                                        n_bronze),
    ("silver has rows",                          n_silver >= DQ_MIN_ROWS,                                        n_silver),
    ("gold has rows",                            n_gold >= DQ_MIN_ROWS,                                          n_gold),
    ("silver: no null Article_ID",               silver.filter(F.col("Article_ID").isNull()).count() == 0,       None),
    ("silver: no null Title",                    silver.filter(F.col("Title").isNull()).count() == 0,            None),
    ("silver: no null Source_name",              silver.filter(F.col("Source_name").isNull()).count() == 0,      None),
    ("silver: no null Url",                      silver.filter(F.col("Url").isNull()).count() == 0,              None),
    ("silver: Article_ID unique",                silver.select("Article_ID").distinct().count() == n_silver,     None),
    ("gold: Article_ID unique",                  gold.select("Article_ID").distinct().count() == n_gold,         None),
    ("gold: contains every silver Article_ID",   silver_ids_missing_from_gold == 0,                               f"{silver_ids_missing_from_gold} missing"),
    ("gold: no orphan Article_IDs absent from silver", gold_ids_orphaned == 0,                                    f"{gold_ids_orphaned} orphaned"),
]

## New: business-aware checks on Gold's trend-scoring math (TR-11 – TR-14)

These assume `04_gold_aggregation` has run at least once against the full 
backlog (`startingVersion=0` on a fresh checkpoint achieves this 
automatically on first deployment) — a `gold.articles` table predating 
this revision, never re-run, would fail the "no null trend_weight" checks 
below through no fault of the math itself.

In [0]:
has_trend_columns = "trend_weight" in gold.columns and "final_trend_score" in gold.columns

if has_trend_columns:
    n_null_trend_weight = gold.filter(F.col("trend_weight").isNull()).count()
    n_null_momentum = gold.filter(F.col("momentum_multiplier").isNull()).count()
    n_null_final_score = gold.filter(F.col("final_trend_score").isNull()).count()

    # TR-11: trend_weight must fall within [0, 1] -- an exponential decay
    # starting at 1.0 (published today) and only ever shrinking. Anything
    # outside this range means the decay formula itself is broken.
    n_trend_weight_out_of_range = gold.filter(
        (F.col("trend_weight") < 0) | (F.col("trend_weight") > 1)
    ).count()

    # TR-12: Engagement, where present, must be non-negative (it's a sum of
    # two COALESCE(...,0)-guarded counts, so a negative value here means
    # something upstream fed it corrupted data).
    n_engagement_negative = gold.filter(F.col("Engagement") < 0).count()

    # TR-12: Engagement must be NULL for editorial sources and NON-null for
    # the three sources that actually have it -- catches a silently broken
    # CASE statement in either direction, not just 'is it in range'.
    engagement_sources = ["dev.to", "Hacker News", "Stack Exchange"]
    n_engagement_sources_missing_value = gold.filter(
        F.col("Source_name").isin(engagement_sources) & F.col("Engagement").isNull()
    ).count()
    n_editorial_sources_with_value = gold.filter(
        (~F.col("Source_name").isin(engagement_sources)) & F.col("Engagement").isNotNull()
    ).count()

    # TR-13: momentum_multiplier must be strictly positive -- a zero or
    # negative multiplier would silently zero out or invert every affected
    # article's trend score.
    n_momentum_not_positive = gold.filter(F.col("momentum_multiplier") <= 0).count()

    # TR-14: final_trend_score must equal trend_weight * momentum_multiplier
    # -- a direct recomputation-and-compare, not just a range check, catches
    # a formula drifting out of sync between 04_gold_aggregation's code and
    # this check over time.
    n_final_score_mismatch = gold.filter(
        F.abs(F.col("final_trend_score") - (F.col("trend_weight") * F.col("momentum_multiplier"))) > F.lit(0.0001)
    ).count()

    # Decay sanity check -- an article clearly older than one half-life
    # (10+ days, comfortably past the 7-day half-life) should have
    # measurably decayed, not still be sitting at the maximum weight. This
    # is what actually catches 'decay looks implemented but is a silent
    # no-op' (e.g. a formula that always evaluates to 1.0).
    old_cutoff = F.date_sub(F.current_date(), 10)
    n_old_articles_undecayed = gold.filter(
        (F.col("Published_date") < old_cutoff) & (F.col("trend_weight") >= 0.999)
    ).count()

    trend_checks = [
        ("gold: no null trend_weight",                       n_null_trend_weight == 0,                     f"{n_null_trend_weight} null"),
        ("gold: no null momentum_multiplier",                 n_null_momentum == 0,                         f"{n_null_momentum} null"),
        ("gold: no null final_trend_score",                   n_null_final_score == 0,                      f"{n_null_final_score} null"),
        ("gold: trend_weight within [0, 1] (TR-11)",           n_trend_weight_out_of_range == 0,             f"{n_trend_weight_out_of_range} out of range"),
        ("gold: Engagement non-negative where present (TR-12)", n_engagement_negative == 0,                  f"{n_engagement_negative} negative"),
        ("gold: Engagement present for dev.to/HN/StackExchange (TR-12)", n_engagement_sources_missing_value == 0, f"{n_engagement_sources_missing_value} missing"),
        ("gold: Engagement NULL for editorial sources (TR-12)", n_editorial_sources_with_value == 0,         f"{n_editorial_sources_with_value} unexpected value(s)"),
        ("gold: momentum_multiplier strictly positive (TR-13)", n_momentum_not_positive == 0,                f"{n_momentum_not_positive} non-positive"),
        ("gold: final_trend_score = trend_weight * momentum_multiplier (TR-14)", n_final_score_mismatch == 0, f"{n_final_score_mismatch} mismatched"),
        ("gold: decay actually applied to old articles (TR-11 sanity check)", n_old_articles_undecayed == 0, f"{n_old_articles_undecayed} undecayed"),
    ]
else:
    log("gold.articles has no trend-scoring columns yet -- run 04_gold_aggregation at least once before this check block applies.")
    trend_checks = []

## Run every check and fail the job on any failure

In [0]:
all_checks = checks + trend_checks
results = [(name, "PASS" if ok else "FAIL", str(detail) if detail is not None else "") for name, ok, detail in all_checks]
display(spark.createDataFrame(results, "check STRING, status STRING, detail STRING"))

failed = [r[0] for r in results if r[1] == "FAIL"]
if failed:
    raise AssertionError(f"Data quality checks failed: {failed}")

log(f"All {len(results)} data quality checks passed")

check,status,detail
bronze has rows,PASS,116
silver has rows,PASS,2731
gold has rows,PASS,2731
silver: no null Article_ID,PASS,
silver: no null Title,PASS,
silver: no null Source_name,PASS,
silver: no null Url,PASS,
silver: Article_ID unique,PASS,
gold: Article_ID unique,PASS,
gold: contains every silver Article_ID,PASS,0 missing


---------------------------------------------------------------------------
AssertionError                            Traceback (most recent call last)
File <command-6277404397853719>, line 7
      5 failed = [r[0] for r in results if r[1] == "FAIL"]
      6 if failed:
----> 7     raise AssertionError(f"Data quality checks failed: {failed}")
      9 log(f"All {len(results)} data quality checks passed")

AssertionError: Data quality checks failed: ['gold: Engagement non-negative where present (TR-12)']